In [86]:
!pip install -q ultralytics

In [87]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image as PILImage
from ultralytics import YOLO
model = YOLO('yolov8n.pt')

In [88]:
cap = cv2.VideoCapture("/content/football_jersey_clip.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
ret,frame = cap.read()
ret2, frame2 = cap.read()

In [89]:
def get_centroids(frame):
  result = model.track(frame, persist=True, verbose=False)
  boxes = result[0].boxes.xyxy.cpu().numpy()
  conf = result[0].boxes.conf.cpu().numpy()
  if result[0].boxes.id is None:
    return []
  ids = result[0].boxes.id.cpu().numpy()
  centroid=[]

  for box,c,tid in zip(boxes,conf,ids):
    if c < 0.5:
      continue
    x1,y1,x2,y2 = box
    cx = (x1+x2)/2
    cy = (y1+y2)/2
    centroid.append((cx,cy,tid))

  return centroid

In [90]:
centroid_frame1 = get_centroids(frame)
centroid_frame2 = get_centroids(frame2)

In [91]:
model = YOLO('yolov8n.pt')
players_position = {}
tid_color ={}
prev_ids = set()
cap = cv2.VideoCapture("/content/football_jersey_clip.mp4")
frame_count = 0
max_frames = 300

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1
    if frame_count > max_frames:
        break

    centroid = get_centroids(frame)
    result = model.track(frame, persist=True, verbose=False)
    boxes = result[0].boxes.xyxy.cpu().numpy()
    ids = result[0].boxes.id.cpu().numpy() if result[0].boxes.id is not None else []
    for box, tid in zip(boxes, ids):
        crop = get_jersey_crop(frame, box)
        tid_color[tid] = get_avg_color(crop)
    curr_ids = { tid for cx, cy, tid in centroid }

    for cx, cy, tid in centroid:
        if tid in players_position:
          players_position[tid].append((cx, cy))

        else:
            players_position[tid] = [(cx, cy)]
        disappeared = prev_ids - curr_ids
    new_ids = curr_ids - prev_ids
    if new_ids:
      for t1 in new_ids:
          for t2 in disappeared:
              dist = np.linalg.norm(tid_color[t1] - tid_color[t2])
              if dist < 15:
                 print("MATCH:", t1, t2)
              print(t1, t2, dist)
    prev_ids = curr_ids

print(players_position)

125.0 119.0 21.57750247809585
119.0 123.0 51.290050602707595
MATCH: 122.0 121.0
122.0 121.0 5.446789411155327
126.0 122.0 15.646387993314596
122.0 118.0 19.55375632922316
130.0 129.0 64.66588511445251
130.0 118.0 63.18515335467183
127.0 118.0 48.66152404861862
142.0 129.0 32.55567972321711
142.0 139.0 15.437850310915117
127.0 129.0 81.51603426718657
146.0 139.0 43.8574117851599
{np.float32(118.0): [(np.float32(403.42545), np.float32(304.4803)), (np.float32(403.38004), np.float32(304.47235)), (np.float32(403.25714), np.float32(304.32013)), (np.float32(403.2357), np.float32(304.36975)), (np.float32(403.01324), np.float32(304.43066)), (np.float32(402.79218), np.float32(304.25397)), (np.float32(402.71887), np.float32(304.17682)), (np.float32(402.5982), np.float32(304.2566)), (np.float32(402.51013), np.float32(304.28696)), (np.float32(402.307), np.float32(304.2952)), (np.float32(402.25714), np.float32(304.13144)), (np.float32(402.2343), np.float32(304.01306)), (np.float32(402.1159), np.floa

In [92]:
players_position = {pid: pos for pid, pos in players_position.items() if len(pos) >= 5}

In [93]:
player_distances = {}

for pid, history in players_position.items():
    total = 0
    for i in range(len(history) - 1):
        x1, y1 = history[i]
        x2, y2 = history[i+1]
        d = ((x2-x1)**2 + (y2-y1)**2)**0.5
        total += d
    player_distances[pid] = total

print(player_distances)

{np.float32(118.0): np.float32(239.69958), np.float32(119.0): np.float32(672.4745), np.float32(120.0): np.float32(255.98813), np.float32(121.0): np.float32(216.69244), np.float32(122.0): np.float32(235.73885), np.float32(123.0): np.float32(50.15697), np.float32(124.0): np.float32(146.47206), np.float32(125.0): np.float32(350.70035), np.float32(126.0): np.float32(92.47485), np.float32(127.0): np.float32(498.4309), np.float32(128.0): np.float32(527.1192), np.float32(129.0): np.float32(554.4177), np.float32(130.0): np.float32(356.23962), np.float32(137.0): np.float32(44.976254), np.float32(138.0): np.float32(46.852142), np.float32(139.0): np.float32(374.2976), np.float32(141.0): np.float32(255.68674), np.float32(142.0): np.float32(294.4292), np.float32(143.0): np.float32(100.93153), np.float32(144.0): np.float32(71.40092), np.float32(145.0): np.float32(79.66818), np.float32(146.0): np.float32(38.863506)}


In [94]:
def get_jersey_crop(frame, box):
    # cast to int since slicing needs whole numbers, not the floats YOLO gives
    x1, y1, x2, y2 = map(int, box)

    height = y2 - y1
    # only take top 35% of box height to isolate jersey, skip shorts/legs
    new_y2 = int(y1 + (0.35 * height))

    # rows (y) first, then columns (x) — standard image slicing order
    return frame[y1:new_y2, x1:x2]

In [95]:

def get_avg_color(crop):
  return crop.mean(axis = (0,1))

In [96]:
avg_colors = []
result = model.track(frame, persist=True, verbose=False)
boxes = result[0].boxes.xyxy.cpu().numpy()
if result[0].boxes.id is None:
    ids = []
else:
    ids = result[0].boxes.id.cpu().numpy()
for box, tid in zip(boxes, ids):
  crop = get_jersey_crop(frame, box)
  avg_color = get_avg_color(crop)
  avg_colors.append((tid, avg_color))
  tid_color[tid] = avg_color

print(len(avg_colors))
print(avg_colors[0])

8
(np.float32(129.0), array([     98.151,      157.89,      137.24]))


In [97]:
colors_only = [c for tid, c in avg_colors]
data = np.array(colors_only, dtype=np.float32)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
compactness, labels, centers = cv2.kmeans(data, 2, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

print(compactness, labels, centers)

837.4863967895508 [[0]
 [0]
 [1]
 [0]
 [0]
 [0]
 [1]
 [0]] [[     93.301      152.81      133.37]
 [     82.538      113.08      128.79]]


In [98]:
player_team = {}
for i, (tid, color) in enumerate(avg_colors):
    if tid not in player_team:
        player_team[tid] = labels[i][0]

print(player_team)

{np.float32(129.0): np.int32(0), np.float32(142.0): np.int32(0), np.float32(143.0): np.int32(1), np.float32(144.0): np.int32(0), np.float32(145.0): np.int32(0), np.float32(146.0): np.int32(0), np.float32(139.0): np.int32(1), np.float32(147.0): np.int32(0)}


In [99]:
team_distance = {}
for pid, dist in player_distances.items():
  if pid not in player_team:
    continue
  team = int(player_team[pid])
  if team not in team_distance:
    team_distance[team]= 0
  team_distance[team] += dist

print(team_distance)


{0: np.float32(1038.7795), 1: np.float32(475.22913)}


In [100]:
import math
def get_player_speeds(position_history, fps):
  speeds = []
  for i in range(1, len(position_history)):
    prev_point = position_history[i-1]
    current_point = position_history[i]
    x1, y1 = prev_point
    x2, y2 = current_point
    distance = math.sqrt((x2-x1)**2 + (y2-y1)**2)
    speed = distance * fps
    speeds.append(speed)
  return speeds

fps = cap.get(cv2.CAP_PROP_FPS)

sample_id = list(players_position.keys())[0]
speeds = get_player_speeds(players_position[sample_id], fps)
print(sample_id)
print(speeds)

118.0
[1.152453821388687, 4.890963902933831, 1.3515198392833099, 5.765800323508536, 7.075194285769812, 2.6605032254823486, 3.61629373392275, 2.329030730611947, 5.082301339002925, 4.279533526807774, 3.013962833756257, 4.373633179711231, 5.426850057782069, 6.460169805378035, 4.591014641692135, 11.513418756215072, 9.798944375631851, 17.233303658430195, 25.9991583389905, 17.85986083497105, 19.658713222808114, 44.67724075461602, 53.943054792821265, 32.46377283009217, 41.62767513087332, 65.27670212348666, 16.25549615474592, 19.377962355107243, 45.359884347923796, 40.20738529741372, 47.99719467260293, 46.45209112983229, 31.456371992538983, 28.738299456008964, 31.322314515503543, 43.976544619915956, 50.873288596795994, 44.663515590370636, 24.363327716234455, 22.749740808708204, 25.255765027811115, 39.986431607077236, 27.695316696704715, 22.052199602759067, 42.93902496061957, 48.16302085672047, 48.57993257582294, 39.454418576628434, 28.5752483085964, 30.908392352689535, 11.62820886731883, 14.57

In [101]:
def count_sprints(speeds, threshold):

  sprint_count = 0
  was_sprinting = False
  for i in speeds:
    is_sprinting = i > threshold
    if is_sprinting and not was_sprinting :
      sprint_count +=1
    was_sprinting = is_sprinting

  return sprint_count

counts = count_sprints(speeds, 30)
print(counts)

38


In [102]:
def build_player_summary(players_position, player_team, player_distances, fps, sprint_threshold):
  player_summary = {}
  for player_id,pos_history in players_position.items():
    # loop over each tracked player to pull together their stats into one summary
    team = player_team.get(player_id, "unknown")
    if not isinstance(team, str):
      team = team.item()
    distance = player_distances[player_id]

    speed = get_player_speeds(pos_history,fps)
    sprint_count = count_sprints(speed,sprint_threshold)

    player_summary[player_id] = {
        "team" : team,
        "distance" : distance,
        "speed" : speed,
        "sprint_count" : sprint_count
    }

  return player_summary


build_player_summary(players_position, player_team, player_distances, fps, 30)


{np.float32(118.0): {'team': 'unknown',
  'distance': np.float32(239.69958),
  'speed': [1.152453821388687,
   4.890963902933831,
   1.3515198392833099,
   5.765800323508536,
   7.075194285769812,
   2.6605032254823486,
   3.61629373392275,
   2.329030730611947,
   5.082301339002925,
   4.279533526807774,
   3.013962833756257,
   4.373633179711231,
   5.426850057782069,
   6.460169805378035,
   4.591014641692135,
   11.513418756215072,
   9.798944375631851,
   17.233303658430195,
   25.9991583389905,
   17.85986083497105,
   19.658713222808114,
   44.67724075461602,
   53.943054792821265,
   32.46377283009217,
   41.62767513087332,
   65.27670212348666,
   16.25549615474592,
   19.377962355107243,
   45.359884347923796,
   40.20738529741372,
   47.99719467260293,
   46.45209112983229,
   31.456371992538983,
   28.738299456008964,
   31.322314515503543,
   43.976544619915956,
   50.873288596795994,
   44.663515590370636,
   24.363327716234455,
   22.749740808708204,
   25.25576502781111

In [103]:
print(tid_color)

{np.float32(118.0): array([     113.56,      153.22,      138.35]), np.float32(119.0): array([     67.244,      122.07,      108.86]), np.float32(120.0): array([     125.32,      150.29,      143.23]), np.float32(121.0): array([     131.29,      161.88,      157.95]), np.float32(122.0): array([     100.64,      154.08,      130.58]), np.float32(123.0): array([     123.63,      157.35,      150.24]), np.float32(124.0): array([     105.34,      130.51,      150.97]), np.float32(125.0): array([     86.863,       111.9,      144.03]), np.float32(126.0): array([     83.633,      122.16,      129.73]), np.float32(127.0): array([     74.773,       99.52,      105.11]), np.float32(128.0): array([     51.344,      91.031,      84.993]), np.float32(129.0): array([     98.151,      157.89,      137.24]), np.float32(130.0): array([     73.688,      107.64,      92.062]), np.float32(137.0): array([     71.923,      124.82,       116.6]), np.float32(138.0): array([     82.773,      121.68,      103.

In [ ]:
print(np.linalg.norm(tid_color[np.float32(53.0)] - tid_color[np.float32(59.0)]))